# 06 — Confidence triage and two-channel AL on one batch

Compare random queries, confidence triage and a two-channel policy using batch 4.
Share evaluation splits and a total budget of 1,000 labels across ten seeds.
High-confidence objects are removed without generating pseudo-labels. Known
oracle labels update classification; 99x labels count as discoveries and never
enter the known-class classifier.


## 0. Configuration


In [ ]:
%load_ext autoreload
%autoreload 2
import sys, os, time
from pathlib import Path
raiz = Path.cwd()
while not (raiz / 'config.py').exists() and raiz != raiz.parent:
    raiz = raiz.parent
assert (raiz / 'config.py').exists(), 'Open this notebook inside the project directory'
os.chdir(raiz); sys.path.insert(0, str(raiz))
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import config
from src.models import classification
from src.models import anomalies as an
from src.models import active_learning as al
from src import pipeline
RS = config.RANDOM_STATE

BATCH    = 4                          
FRACS    = (0.0, 0.3, 0.6, 1.0)
SEEDS = tuple(range(10))           
RECOMPUTE_FINAL_RESULTS = False  
print('config · batch:', BATCH, '· fracs:', FRACS, '· seeds:', SEEDS)


## 1. Load batch 4 and build the model factory


In [ ]:
tr = classification.features_e2gp('train')
X_tr, COLS = classification.e2gp_matrix(tr); y_tr = tr['target'].to_numpy()

batch_paths = pipeline.select_batch(BATCH)
te = classification.features_e2gp('test', batch=BATCH)
metadata_table = pd.read_csv(batch_paths.test_metadata_clean_csv)
te = te.merge(metadata_table[['object_id', 'true_target']], on='object_id')
tt = te['true_target'].to_numpy()
X_te, _ = classification.e2gp_matrix(te, COLS)

RETRAIN_MODEL = False
if RETRAIN_MODEL:
    clf0 = classification.e2gp_al(y=y_tr, random_state=RS).fit(X_tr, y_tr)
else:
    clf0, saved_columns, model_contract = classification.load_e2gp_artifact()
    if list(COLS) != list(saved_columns):
        raise ValueError('Current features do not match the saved model contract')
    print('Loaded the verified canonical model; classifier fitting was not repeated.')
gp_te = te[['object_id']].merge(pd.read_csv(batch_paths.features_gp_test_csv), on='object_id', how='left')
st_te = te[['object_id']].merge(pd.read_csv(batch_paths.features_test_csv), on='object_id', how='left')
S = np.column_stack([an.confidence_novelty_score(clf0.predict_proba(X_te)),
                     an.score_isolation_forest(X_te),
                     an.gp_residual_score(gp_te),
                     an.amplitude_score(st_te)])
desc_te = S[:, 1:]

con = tt < 990
v, c = np.unique(tt[con], return_counts=True); singles = set(v[c < 2].tolist())
keep = ~(con & np.isin(tt, list(singles)))
if singles:
    print(f'eliminados {int((~keep).sum())} object(s) of classes singleton {sorted(singles)}')
X_te, tt, desc_te = X_te[keep], tt[keep], desc_te[keep]

def make_model(y_lab):
    return classification.e2gp_al(y=y_lab, random_state=RS)

print(f'batch {BATCH}: {len(tt):,} objects · {int((tt>=990).sum())} anomalies 99x')


## 1b. OOF calibration and operating thresholds

Estimate calibration inside the outer CV training folds. The current saved
probabilities give ECE 0.2172 → 0.0669 and multiclass Brier score
0.3363 → 0.2618. The thresholds 0.30 and 0.60 are design choices, not universal
optima. Known-class calibration does not guarantee detection of unknown classes.


In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from src.models import evaluation as ev

CACHE_CAL = config.DIR_PROCESSED / 'calibracion_e2gp_oof_150_cv3.npz'
CLASES = np.sort(np.unique(y_tr))
if CACHE_CAL.exists():
    z = np.load(CACHE_CAL)
    proba_sin, proba_cal = z['proba_sin'], z['proba_cal']
    if len(proba_cal) != len(y_tr) or not np.array_equal(z['clases'], CLASES):
        raise ValueError('Cache of calibration incompatible; elimina the NPZ and reejecuta the celda')
else:
    cv_oof = StratifiedKFold(5, shuffle=True, random_state=RS)
    rf_sin = RandomForestClassifier(
        n_estimators=150, min_samples_leaf=3, class_weight='balanced',
        random_state=RS, n_jobs=-1)
    e2gp_cal = classification.e2gp_al(y=y_tr, random_state=RS)
    proba_sin = cross_val_predict(
        rf_sin, X_tr, y_tr, cv=cv_oof, method='predict_proba', n_jobs=1)
    proba_cal = cross_val_predict(
        e2gp_cal, X_tr, y_tr, cv=cv_oof, method='predict_proba', n_jobs=1)
    np.savez_compressed(CACHE_CAL, proba_sin=proba_sin, proba_cal=proba_cal, clases=CLASES)

ece_sin, curva_sin = ev.confidence_ece(y_tr, proba_sin, CLASES)
ece_cal, curva_cal = ev.confidence_ece(y_tr, proba_cal, CLASES)
brier_sin = ev.multiclass_brier_score(y_tr, proba_sin, CLASES)
brier_cal = ev.multiclass_brier_score(y_tr, proba_cal, CLASES)

fig, (ax, axh) = plt.subplots(1, 2, figsize=(11.2, 4.4), gridspec_kw={'width_ratios': [1.15, 1]})
for light_curve, etiqueta, color, marker in [
        (curva_sin, f'Uncalibrated · ECE={ece_sin:.3f}', 'C0', 'o'),
        (curva_cal, f'Platt cv=3 · ECE={ece_cal:.3f}', 'C1', 's')]:
    x = [p[0] for p in light_curve]; y = [p[1] for p in light_curve]
    ax.plot(x, y, marker=marker, lw=2, label=etiqueta, color=color)
ax.plot([0, 1], [0, 1], '--', color='0.45', lw=1.2, label='Perfect calibration')
ax.set(xlim=(0, 1), ylim=(0, 1), xlabel='Mean confidence',
       ylabel='Empirical accuracy', title='E2-GP OOF reliability')
ax.legend(fontsize=8); ax.grid(alpha=.25)
conf_cal = proba_cal.max(axis=1)
axh.hist(conf_cal, bins=np.linspace(0, 1, 21), color='C1', alpha=.75)
for u, text_value, color in [(0.30, 'novedad', 'C3'), (0.60, 'auto-clasificación', 'C2')]:
    axh.axvline(u, color=color, ls='--', lw=1.8, label=f'{u:.2f} · {text_value}')
axh.set(xlabel='Calibrated OOF confidence', ylabel='Number of objects',
        title='Triage operating regions')
axh.legend(fontsize=8); axh.grid(alpha=.25)
fig.suptitle('Canonical E2-GP: 150-tree RF + three-fold Platt calibration', fontweight='bold')
fig.tight_layout(); fig.savefig(config.FIGURE_DIR / 'calibracion_E2GP_fiabilidad.png', dpi=180, bbox_inches='tight'); plt.show()
display(pd.DataFrame({
    'modelo': ['without calibrar', 'Platt cv=3'],
    'ECE': [ece_sin, ece_cal], 'Brier_multiclase': [brier_sin, brier_cal]}).round(4))


### Calibration interpretation

Use the metrics computed above, with the same OOF probability cache. Calibration
improves internal reliability but does not remove domain shift or prevent 99x
objects from receiving high known-class confidence.


## 1c. Exploratory comparison of query criteria


In [ ]:
CSV_ESTR = config.DIR_PROCESSED / 'estrategias_consulta_batch4_5seeds.csv'
if CSV_ESTR.exists():
    estr = pd.read_csv(CSV_ESTR)
    print(f'Cache loaded: {CSV_ESTR.name}')
else:
    estr = al.compare_query_strategies_multiseed(
        X_tr, y_tr, X_te, tt, make_model,
        strategies_list=('least_confidence', 'margin', 'entropy', 'random'),
        random_seeds=tuple(range(5)), test_size_eval=0.2, b=100, n_iter=10,
        output_path=str(CSV_ESTR), verbose=True)
    print(f'saved: {CSV_ESTR.name}')
fin_estr = estr.sort_values('iter').groupby(['estrategia', 'semilla']).tail(1)
sort_order = ['entropy', 'least_confidence', 'margin', 'random']
class_labels = ['Entropy', '1 − confidence', 'Margin', 'Random']
res_estr = fin_estr.groupby('estrategia').agg(
    recall=('recall_anom', 'mean'), recall_sd=('recall_anom', 'std'),
    MF1=('MF1', 'mean'), MF1_sd=('MF1', 'std'), n=('semilla', 'nunique')).loc[sort_order]
res_estr['recall_ee'] = res_estr['recall_sd'] / np.sqrt(res_estr['n'])
res_estr['MF1_ee'] = res_estr['MF1_sd'] / np.sqrt(res_estr['n'])

fig, axes = plt.subplots(1, 2, figsize=(10.8, 4.2))
colores = ['#2a9d8f', '#2166ac', '#c44e52', '#8d8d8d']
x = np.arange(len(sort_order))
for ax, metric_name, error, ylabel, plot_title in [
        (axes[0], 'recall', 'recall_ee', 'Recall of anomalies', 'Anomaly discovery'),
        (axes[1], 'MF1', 'MF1_ee', 'Macro-F1', 'Classification')]:
    vals = res_estr[metric_name].to_numpy(); errs = res_estr[error].to_numpy()
    ax.bar(x, vals, yerr=errs, capsize=4, color=colores, width=.72)
    ax.set_xticks(x); ax.set_xticklabels(class_labels, rotation=18, ha='right')
    ax.set_ylabel(ylabel); ax.set_title(plot_title); ax.grid(axis='y', alpha=.25)
    for xi, v, e in zip(x, vals, errs): ax.text(xi, v + e + .01 * max(vals), f'{v:.3f}', ha='center', fontsize=8)
fig.suptitle('Query-strategy comparison · batch 4 · 5 seeds', fontweight='bold')
fig.tight_layout(); fig.savefig(config.FIGURE_DIR / 'al_estrategias_consulta_batch4.png', dpi=180, bbox_inches='tight'); plt.show()
display(res_estr[['recall', 'recall_ee', 'MF1', 'MF1_ee', 'n']].round(4))


### Query-criterion interpretation

Compare entropy, least confidence, margin and random queries on five shared splits.
Entropy is the selected design criterion. This exploratory comparison is not an
independent confirmation of the final policy's performance.


## 2. Two channels: ten seeds and several discovery allocations

Reuse the saved trajectories when they exist and `RECOMPUTE_FINAL_RESULTS` is
false. Otherwise the following cell launches the expensive AL experiment.


In [ ]:
CSV = config.DIR_PROCESSED / 'dos_canales_batch4_10seeds.csv'
if CSV.exists() and not RECOMPUTE_FINAL_RESULTS:
    hist = pd.read_csv(CSV)
    print(f'Cache loaded: {CSV.name}')
else:
    t = time.time()
    hist = al.two_channel_multiseed(
        X_tr, y_tr, X_te, tt, desc_te, make_model,
        fracs=FRACS, random_seeds=SEEDS, conf_dudoso=0.60, reg_C=0.1,
        budget=1000, b_iter=100, n_iter=30, output_path=str(CSV), verbose=True)
    print(f'\ndos canales (batch {BATCH}, 10 seeds) hecho in {(time.time()-t)/60:.1f} min')

fin = hist.sort_values('iter').groupby(['frac_desc', 'semilla']).tail(1)
summary_table = (fin.groupby('frac_desc')
           .agg(MF1_mean=('MF1', 'mean'), MF1_std=('MF1', 'std'),
                descub_mean=('n_descubiertas', 'mean'), descub_std=('n_descubiertas', 'std'),
                descub_desc=('descub_desc', 'mean'), descub_clas=('descub_clas', 'mean')).round(3))
display(summary_table)


## 3. Confidence triage and random queries on the same ten seeds


In [ ]:
CSV_TRI = config.DIR_PROCESSED / 'triaje_batch4_10seeds.csv'
if CSV_TRI.exists() and not RECOMPUTE_FINAL_RESULTS:
    hist_tri = pd.read_csv(CSV_TRI)
    print(f'Cache loaded: {CSV_TRI.name}')
else:
    t = time.time()
    hist_tri = al.confidence_triage_multiseed(
        X_tr, y_tr, X_te, tt, make_model,
        modes_list=('triage', 'random'), random_seeds=SEEDS,
        conf_anom=0.30, conf_dudoso=0.60, orden_dudoso='entropy',
        budget=1000, b_iter=100, n_iter=30, output_path=str(CSV_TRI), verbose=True)
    print(f'\ntriaje+random (batch {BATCH}, 10 seeds) hecho in {(time.time()-t)/60:.1f} min')

fin_tri = hist_tri.sort_values('iter').groupby(['modo', 'semilla']).tail(1)
resumen_tri = (fin_tri.groupby('modo')
               .agg(MF1_mean=('MF1', 'mean'), MF1_std=('MF1', 'std'),
                    descub_mean=('n_descubiertas', 'mean'), descub_std=('n_descubiertas', 'std'),
                    q_anom=('q_anom', 'mean'), n_auto=('n_auto', 'mean')).round(3))
print('Triage by confidence and random (baseline) · 10 seeds:')
display(resumen_tri)


## 4. Final-state comparison


In [ ]:
records_list = []
for m in ['triage', 'random']:
    if m in resumen_tri.index:
        etq = 'Triage by confidence' if m == 'triage' else 'random (baseline)'
        records_list.append({'sistema': etq, 'MF1': resumen_tri.loc[m, 'MF1_mean'],
                      'MF1_std': resumen_tri.loc[m, 'MF1_std'],
                      'descub': resumen_tri.loc[m, 'descub_mean'], 'descub_std': resumen_tri.loc[m, 'descub_std']})
for fr in summary_table.index:
    records_list.append({'sistema': f'Two channels (frac_desc={fr})', 'MF1': summary_table.loc[fr, 'MF1_mean'],
                  'MF1_std': summary_table.loc[fr, 'MF1_std'],
                  'descub': summary_table.loc[fr, 'descub_mean'], 'descub_std': summary_table.loc[fr, 'descub_std']})
comp = pd.DataFrame(records_list)
comp.to_csv(config.DIR_PROCESSED / 'comparacion_batch4_10seeds.csv', index=False)
print(f'Batch {BATCH} · 10 seeds · mean ± standard deviation:')
display(comp)

x = list(summary_table.index)
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6))
axes[0].errorbar(x, summary_table['MF1_mean'], yerr=summary_table['MF1_std'], marker='o', capsize=3, label='Two channels')
axes[1].errorbar(x, summary_table['descub_mean'], yerr=summary_table['descub_std'], marker='o', capsize=3, color='C3', label='Two channels')
for m, c in [('triage', 'gray'), ('random', 'C7')]:
    if m in resumen_tri.index:
        lab = 'Triage by confidence' if m == 'triage' else 'random'
        axes[0].axhline(resumen_tri.loc[m, 'MF1_mean'], ls='--', color=c, lw=1.2, label=lab)
        axes[1].axhline(resumen_tri.loc[m, 'descub_mean'], ls='--', color=c, lw=1.2, label=lab)
axes[0].set_title('MF1 vs frac_desc (10 seeds)'); axes[0].set_xlabel('frac_desc'); axes[0].set_ylabel('MF1'); axes[0].grid(alpha=.3); axes[0].legend(fontsize=8)
axes[1].set_title('Anomaly discovery vs frac_desc (10 seeds)'); axes[1].set_xlabel('frac_desc'); axes[1].set_ylabel('Discovered anomalies'); axes[1].grid(alpha=.3); axes[1].legend(fontsize=8)
fig.suptitle(f'Batch {BATCH} · two channels vs triage by confidence vs random sampling · 10 seeds'); fig.tight_layout(); plt.show()


## 4b. Standard errors and paired comparisons


In [ ]:
from scipy.stats import wilcoxon
n = len(SEEDS)
comp['MF1_ee'] = (comp['MF1_std'] / np.sqrt(n)).round(4)
comp['descub_ee'] = (comp['descub_std'] / np.sqrt(n)).round(2)
print('Table with standard errors (std/√n):')
display(comp[['sistema', 'MF1', 'MF1_ee', 'descub', 'descub_ee']])

tri = fin_tri[fin_tri['modo'] == 'triage'].set_index('semilla')
print(f'\nWilcoxon emparejado (1-cola: two channels > triage by confidence), n={n} seeds:')
for fr in sorted(fin['frac_desc'].unique()):
    dfr = fin[fin['frac_desc'] == fr].set_index('semilla')
    text_line = f'  frac={fr}:'
    for metric_name, col in [('MF1', 'MF1'), ('descub', 'n_descubiertas')]:
        a = dfr.loc[tri.index, col].values; b = tri[col].values
        try:
            _, pv = wilcoxon(a, b, alternative='greater'); pv = f'{pv:.4f}'
        except Exception:
            pv = 'n/a'
        text_line += f'  [{metric_name}] gana {int((a>b).sum())}/{n}, p={pv}'
    print(text_line)


## 4c. Learning curves


In [ ]:
dc = pd.read_csv(config.DIR_PROCESSED / 'dos_canales_batch4_10seeds.csv')
trj = pd.read_csv(config.DIR_PROCESSED / 'triaje_batch4_10seeds.csv')
curvas = pd.concat([
    dc[np.isclose(dc['frac_desc'], 0.6)].assign(sistema='two channels'),
    trj[trj['modo'] == 'triage'].assign(sistema='triage'),
    trj[trj['modo'] == 'random'].assign(sistema='random')], ignore_index=True)
res = curvas.groupby(['sistema', 'n_consultas']).agg(
    recall=('recall_anom', 'mean'), recall_sd=('recall_anom', 'std'),
    MF1=('MF1', 'mean'), MF1_sd=('MF1', 'std'), n=('semilla', 'nunique')).reset_index()
res['recall_ee'] = res['recall_sd'] / np.sqrt(res['n'])
res['MF1_ee'] = res['MF1_sd'] / np.sqrt(res['n'])

pal = {'two channels': '#2166ac', 'triage': '#d6604d', 'random': '#8d8d8d'}
fig, axes = plt.subplots(1, 2, figsize=(11.5, 4.4))
for ax, metric_name, error, ylabel, plot_title in [
        (axes[0], 'recall', 'recall_ee', 'Recall of anomalies', 'Anomaly discovery'),
        (axes[1], 'MF1', 'MF1_ee', 'Macro-F1', 'Classification')]:
    for sistema in ['two channels', 'triage', 'random']:
        d = res[res['sistema'] == sistema].sort_values('n_consultas')
        x = d['n_consultas'].to_numpy(); y = d[metric_name].to_numpy(); e = d[error].to_numpy()
        ax.plot(x, y, marker='o', ms=4, lw=2, color=pal[sistema], label=sistema)
        ax.fill_between(x, y-e, y+e, color=pal[sistema], alpha=.16)
    ax.set_xlabel('Cumulative expert queries'); ax.set_ylabel(ylabel)
    ax.set_title(plot_title); ax.grid(alpha=.25); ax.legend(fontsize=8)
axes[0].set_ylim(bottom=0)
fig.suptitle('Label efficiency in batch 4 · 10 seeds', fontweight='bold')
fig.tight_layout(); fig.savefig(config.FIGURE_DIR / 'al_curvas_aprendizaje_batch4.png', dpi=180, bbox_inches='tight'); plt.show()
fin_curvas = curvas.sort_values('iter').groupby(['sistema', 'semilla']).tail(1)
display(fin_curvas.groupby('sistema')[['recall_anom', 'MF1']].agg(['mean', 'std']).round(4))


### Learning-curve interpretation

The curves measure discoveries and classification as labels are acquired.
In the reference batch the active policies outperform random queries, while
their discovery totals are similar. Similar observed scores are not a formal
equivalence or non-inferiority result.


## Scope of the single-batch result

The result is conditional on this simulated batch, its class mixture, full
light curves and an ideal oracle. Evaluate other batches and the larger pool
before attributing a universal advantage to a policy.
